In [1]:
import torch
import os
import joblib
import pickle
import numpy as np
import pandas as pd
from Bio.PDB.MMCIF2Dict import MMCIF2Dict
from Bio.SeqUtils import seq1
from tqdm.notebook import tqdm

In [2]:
def weighted_sum(s_inputs, s, z, weight, interface):
    residues = torch.where(interface)
    weight = weight[residues]
    weight_sum = weight.sum()

    si_i = s_inputs[residues[0]]
    si_j = s_inputs[residues[1]]
    s_i = s[residues[0]]
    s_j = s[residues[1]]

    wsi = (si_i.unsqueeze(-1) * si_j.unsqueeze(-2) * weight[:, None, None] / weight_sum).sum(0)
    ws = (s_i.unsqueeze(-1) * s_j.unsqueeze(-2) * weight[:, None, None] / weight_sum).sum(0)
    wz = (z[residues] * weight[:, None] / weight_sum).sum(0)

    return (wsi, ws, wz)

In [3]:
def load_data(path, subpath_list, chain_ids, *, subpath_map=None, identifier_map=None, model_id=0):
    data_list = []
    exception = []
    distogram_bins = torch.arange(64) * 0.3 + 2.15
    iterator = enumerate(subpath_list) if len(subpath_list) == 1 else enumerate(tqdm(subpath_list))

    for i, d in iterator:
        try:
            data = { 'subpath': d }
            p = d if subpath_map is None else subpath_map(d)
            f = d if identifier_map is None else identifier_map(d)
    
            conf_path = f'{path}/{p}/confidence_{f}_model_{model_id}.json'
            cif_path = f'{path}/{p}/{f}_model_{model_id}.cif'
            ci_path = f'{path}/{p}/confidence_inputs_{f}_model_{model_id}.pkl'

            with open(conf_path) as f:
                conf = json.load(f)
    
            data['confidence_score'] = conf['confidence_score']
            data['iptm'] = conf['iptm']
            data['iplddt'] = conf['complex_iplddt']

            cif = MMCIF2Dict(cif_path)
            strand_labels = cif['_entity_poly.pdbx_strand_id']
            strand_seqs = cif['_entity_poly.pdbx_seq_one_letter_code']
            sequence = ''
            for k, strand_seq in enumerate(strand_seqs):
                seq = strand_seq.replace('(', '').replace(')', '').replace('\n', '')
                seq = seq1(seq)
                sequence += seq * len(strand_labels[k].split(','))
    
            strand_ids = cif['_pdbx_poly_seq_scheme.pdb_strand_id']
            chains = chain_ids[i]
            data['chain_id'] = chains
    
            chain_enum = {}
            for k in range(len(chains)):
                for c in chains[k]:
                    chain_enum[c] = k
    
            asym_ids = torch.tensor(list(map(lambda t: chain_enum[t] if t in chain_enum else -1, strand_ids)), dtype=torch.bfloat16)
            c = (asym_ids >= 0)
            interchain = c[:, None] & c[None, :] & (asym_ids[:, None] != asym_ids[None, :])

            with open(ci_path, 'rb') as f:
                ci = pickle.load(f)
    
            s_inputs = ci['s_inputs'].squeeze(0)
            s = ci['s'].squeeze(0)
            z = ci['z'].squeeze(0)
            distogram = ci['pred_distogram_logits'].squeeze(0)
            distance = (torch.softmax(distogram, dim=2) * distogram_bins[None, None, :]).sum(dim=2)
            weight = 1 / distance ** 2
    
            threshold = max(12, torch.quantile(distance[interchain], .01))
            interface = ((distance <= threshold) * interchain)
    
            data['wsi'], data['ws'], data['wz'] = weighted_sum(s_inputs, s, z, weight, interface)
            exception.append('')
            
        except Exception as e:
            exception.append(e.__str__())
            continue

        data_list.append(data)

    return data_list, exception

## SKEMPI

In [4]:
skempi_df = pd.read_csv('data/skempi_v2.csv', sep=';')
skempi_df

,#Pdb,Mutation(s)_PDB,Mutation(s)_cleaned,iMutation_Location(s),Hold_out_type,Hold_out_proteins,Affinity_mut (M),Affinity_mut_parsed,Affinity_wt (M),Affinity_wt_parsed,...,koff_mut_parsed,koff_wt (s^(-1)),koff_wt_parsed,dH_mut (kcal mol^(-1)),dH_wt (kcal mol^(-1)),dS_mut (cal mol^(-1) K^(-1)),dS_wt (cal mol^(-1) K^(-1)),Notes,Method,SKEMPI version
0,1CSE_E_I,LI45G,LI38G,COR,Pr/PI,Pr/PI,5.26E-11,5.260000e-11,1.12E-12,1.120000e-12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IASP,1
1,1CSE_E_I,LI45S,LI38S,COR,Pr/PI,Pr/PI,8.33E-12,8.330000e-12,1.12E-12,1.120000e-12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IASP,1
2,1CSE_E_I,LI45P,LI38P,COR,Pr/PI,Pr/PI,1.02E-07,1.020000e-07,1.12E-12,1.120000e-12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IASP,1
3,1CSE_E_I,LI45I,LI38I,COR,Pr/PI,Pr/PI,1.72E-10,1.720000e-10,1.12E-12,1.120000e-12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IASP,1
4,1CSE_E_I,LI45D,LI38D,COR,Pr/PI,Pr/PI,1.92E-09,1.920000e-09,1.12E-12,1.120000e-12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,IASP,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
7080,3QIB_ABP_CD,KP9R,KP8R,COR,TCR/pMHC,"TCR/pMHC,1JCK_A_B",2.4E-04,2.400000e-04,5.5E-06,5.500000e-06,...,0.500,2.2E-02,0.022,NaN,NaN,NaN,NaN,NaN,SPR,2
7081,3QIB_ABP_CD,TP12A,TP11A,COR,TCR/pMHC,"TCR/pMHC,1JCK_A_B",>1.1E-03,1.100000e-03,5.5E-06,5.500000e-06,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,SPR,2
7082,3QIB_ABP_CD,TP12S,TP11S,COR,TCR/pMHC,"TCR/pMHC,1JCK_A_B",3.38E-05,3.380000e-05,5.5E-06,5.500000e-06,...,0.134,2.2E-02,0.022,NaN,NaN,NaN,NaN,NaN,SPR,2
7083,3QIB_ABP_CD,TP12N,TP11N,COR,TCR/pMHC,"TCR/pMHC,1JCK_A_B",4.34E-05,4.340000e-05,5.5E-06,5.500000e-06,...,0.175,2.2E-02,0.022,NaN,NaN,NaN,NaN,NaN,SPR,2


In [5]:
method_tiers = {
    1: ['SPR', 'ITC', 'BI', 'KinExA',],
    2: ['CSPRIA', 'ELISA', 'ELFA', 'SFFL', 'SFPP', 'SE', 'SP', 'RA', 'ESMA', 'IARA', 'IAGE', 'IAFL',],
    3: ['IASP', 'EMSA', 'SPR,SFFL', 'FL']
}
priority = {}
for i in method_tiers:
    for method in method_tiers[i]:
        priority[method] = i

skempi_wt = skempi_df[~np.isnan(skempi_df['Affinity_wt_parsed'])].copy()
max_version = skempi_wt.groupby('#Pdb')['SKEMPI version'].transform('max')
skempi_wt = skempi_wt[skempi_wt['SKEMPI version'] == max_version]
skempi_wt['Method_priority'] = skempi_wt['Method'].map(priority).fillna(4)
first_priority = skempi_wt.groupby('#Pdb')['Method_priority'].transform('min')
skempi_wt = skempi_wt[skempi_wt['Method_priority'] == first_priority]
skempi_wt = skempi_wt[['#Pdb', 'Affinity_wt_parsed']].groupby('#Pdb').median()

skempi_mut = skempi_df[~np.isnan(skempi_df['Affinity_mut_parsed'])].copy()
skempi_mut['subpath'] = skempi_mut['#Pdb'] + '_' + skempi_mut['Mutation(s)_cleaned'].map(lambda t: ','.join(sorted(t.split(','))))
max_version = skempi_mut.groupby('subpath')['SKEMPI version'].transform('max')
skempi_mut = skempi_mut[skempi_mut['SKEMPI version'] == max_version]
skempi_mut['Method_priority'] = skempi_mut['Method'].map(priority).fillna(4)
first_priority = skempi_mut.groupby('subpath')['Method_priority'].transform('min')
skempi_mut = skempi_mut[skempi_mut['Method_priority'] == first_priority]
skempi_mut['ddG'] = (8.314 / 4184) * (273.15 + 25.0) * np.log(skempi_mut['Affinity_mut_parsed'] / skempi_mut['Affinity_wt_parsed'])
skempi_mut = skempi_mut[['subpath', 'Affinity_mut_parsed', 'ddG']].groupby('subpath').median()
skempi_mut['#Pdb'] = skempi_mut.index.map(lambda t: '_'.join(t.split('_')[:3]))

In [6]:
pkl_path = 'processed_data/skempi_mut.pkl'
boltz_path = 

In [7]:
if os.path.exists(pkl_path):
    skempi = joblib.load(pkl_path)
    
else:
    ppba = {}
    
    for d in os.listdir(boltz_path):
        if d == 'confidence_scores.tsv':
            continue
            
        # { d = boltz_results_1A22_A_B_DA160A,RA64A }
        w = d.split('_')
        pdb = w[2]
        chains = ''.join(w[3:5])
        mutstr = w[5]
        
        chains = ''.join(sorted(list(chains)))
        mutstr = ','.join(sorted(mutstr.split(',')))
        
        if pdb not in ppba:
            ppba[pdb] = {}
        if chains not in ppba[pdb]:
            ppba[pdb][chains] = {}
    
        assert mutstr not in ppba[pdb][chains]
        ppba[pdb][chains][mutstr] = d
    
    subpath_list = []
    chain_ids = []
    skempi_index = []
    
    for t in skempi_mut.index:
        w = t.split('_')
        pdb = w[0]
        chains = ''.join(w[1:3])
        mutstr = ','.join(sorted(w[3:]))
    
        if pdb not in ppba:
            continue
    
        for chains_list in ppba[pdb]:
            if not set(list(chains)).issubset(set(list(chains_list))):
                continue
    
            if mutstr in ppba[pdb][chains_list]:
                subpath_list.append(ppba[pdb][chains_list][mutstr])
                chain_ids.append(w[1:3])
                skempi_index.append(t)

    data_list, e = load_data(boltz_path, subpath_list, chain_ids,
                             subpath_map=lambda t: f'{t}/predictions/{'_'.join(t.split('_')[2:])}',
                             identifier_map=lambda t: '_'.join(t.split('_')[2:]))
    valid_index = np.where(np.array(e) == '')[0]
    skempi_index = np.array(skempi_index)[valid_index]

    df = pd.merge(skempi_mut, skempi_wt, left_on='#Pdb', right_index=True)
    df['Affinity_mut_normalized'] = np.exp(df['ddG'] / ((8.314 / 4184) * (273.15 + 25.0))) * df['Affinity_wt_parsed']    
    labels = torch.from_numpy(np.log10(1e9 * df.loc[skempi_index]['Affinity_mut_normalized'].values)).float()
    
    skempi = (data_list, labels)
    joblib.dump(skempi, pkl_path, compress=3)

In [8]:
pkl_path = 'processed_data/skempi_wt.pkl'
boltz_path = 

In [9]:
if os.path.exists(pkl_path):
    wild_types = joblib.load(pkl_path)
    
else:
    ppba = {}
    
    for d in os.listdir(boltz_path):
        if d == 'confidence_scores.tsv':
            continue

        # { d = boltz_results_1A22_A_B }
        w = d.split('_')
        pdb = w[2]
        chains = ''.join(w[3:5])
        chains = ''.join(sorted(list(chains)))
        
        if pdb not in ppba:
            ppba[pdb] = {}
    
        ppba[pdb][chains] = d
        
    subpath_list = []
    chain_ids = []
    skempi_index = []
    
    for t in skempi_wt.index:
        w = t.split('_')
        pdb = w[0]
        chains = ''.join(w[1:3])
    
        if pdb not in ppba:
            continue
    
        for chains_list in ppba[pdb]:
            if not set(list(chains)).issubset(set(list(chains_list))):
                continue
            if ppba[pdb][chains_list] in subpath_list:
                continue
                
            subpath_list.append(ppba[pdb][chains_list])
            chain_ids.append(w[1:3])
            skempi_index.append(t)

    data_list, e = load_data(boltz_path, subpath_list, chain_ids,
                             subpath_map=lambda t: f'{t}/predictions/{'_'.join(t.split('_')[2:])}',
                             identifier_map=lambda t: '_'.join(t.split('_')[2:]))
    valid_index = np.where(np.array(e) == '')[0]
    skempi_index = np.array(skempi_index)[valid_index]
    
    labels = torch.from_numpy(np.log10(1e9 * skempi_wt.loc[skempi_index]['Affinity_wt_parsed'].values)).float()
    
    wild_types = (data_list, labels)
    joblib.dump(wild_types, pkl_path, compress=3)

## Her2

In [10]:
her2_df = pd.read_csv('data/HER2_binders.csv').fillna('')
her2_df.index = '1n8z_HL_A_' + her2_df['mutation']
her2_df

,KD (nM),mutation,num_mutations,ddG,pdb_fname,chain_a,chain_b
mutation,,,,,,,
1n8z_HL_A_,1.94,,0,0.000000,1n8z.pdb,HL,A
"1n8z_HL_A_SH105A,DH110Y",0.56,"SH105A,DH110Y",2,-0.723782,1n8z.pdb,HL,A
"1n8z_HL_A_SH105T,GH108Y,GH109Y,DH110F,GH111I,FH112Y,AH114Y,YH117V",0.87,"SH105T,GH108Y,GH109Y,DH110F,GH111I,FH112Y,AH11...",8,-0.462774,1n8z.pdb,HL,A
"1n8z_HL_A_SH105T,WH107Y,GH108F,GH109F,DH110N,FH112W,AH114Y,MH115F,YH117V",0.94,"SH105T,WH107Y,GH108F,GH109F,DH110N,FH112W,AH11...",9,-0.416926,1n8z.pdb,HL,A
"1n8z_HL_A_SH105A,WH107Y,GH108Y,DH110Y,FH112G,AH114W,MH115F,DH116A",0.98,"SH105A,WH107Y,GH108Y,DH110Y,FH112G,AH114W,MH11...",8,-0.392237,1n8z.pdb,HL,A
...,...,...,...,...,...,...,...
"1n8z_HL_A_SH105A,WH107S,DH110S,FH112Y,AH114Y,MH115Y",1830.41,"SH105A,WH107S,DH110S,FH112Y,AH114Y,MH115Y",6,4.070407,1n8z.pdb,HL,A
"1n8z_HL_A_SH105A,WH107Y,GH108S,GH109S,DH110S,FH112G,YH113S,MH115F",1986.37,"SH105A,WH107Y,GH108S,GH109S,DH110S,FH112G,YH11...",8,4.118851,1n8z.pdb,HL,A
"1n8z_HL_A_SH105A,WH107V,DH110R,FH112Y,MH115Y",2286.40,"SH105A,WH107V,DH110R,FH112Y,MH115Y",5,4.202191,1n8z.pdb,HL,A


In [11]:
pkl_path = 'processed_data/her2.pkl'
boltz_path = 

In [12]:
if os.path.exists(pkl_path):
    her2 = joblib.load(pkl_path)
    
else:
    subpath_list = []
    chain_ids = []
    her2_index = []

    for d in os.listdir(boltz_path):
        if d == 'confidence_scores.tsv':
            continue

        # { d = boltz_results_1n8z_HL_A_DH110P,FH112L }
        w = d.split('_')
        chains = w[3:5]
        mutstr = w[5]

        subpath_list.append(d)
        chain_ids.append(chains)
        her2_index.append('_'.join(w[2:]))

    data_list, e = load_data(boltz_path, subpath_list, chain_ids,
                             subpath_map=lambda t: f'{t}/predictions/{'_'.join(t.split('_')[2:])}',
                             identifier_map=lambda t: '_'.join(t.split('_')[2:]))
    valid_index = np.where(np.array(e) == '')[0]
    her2_index = np.array(her2_index)[valid_index]
    
    labels = torch.from_numpy(np.log10(1e9 * her2_df.loc[her2_index]['KD (nM)'].values)).float()
    
    her2 = (data_list, labels)
    joblib.dump(her2, pkl_path, compress=3)